# Data-like vs Prediction-like Event 3D Visualization

This notebook generates comprehensive 3D visualizations comparing data-like events (using real photon trajectories from ROOT files) with prediction-like events (generated from physics simulation) using the same track parameters.

## Features:
- **Scatter-based plots**: Interactive 3D scatter plots with track visualization
- **Disc-based plots**: Using the detector's native visualization method
- **Statistical analysis**: Comprehensive comparison of event characteristics
- **Flexible configuration**: Multiple detectors, entries, and visualization options

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
import plotly.subplots as sp
from datetime import datetime

# LUCiD imports
from tools.geometry import generate_detector
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim
from tools.utils import spherical_to_cartesian, base_dir_path
from tools.optimization.event_cache import get_detector_bounds, generate_random_event_params

## Configuration Parameters

Adjust these parameters to customize the visualization:

In [ ]:
# Configuration parameters
CONFIG = {
    'detector_config': base_dir_path() + 'config/IWCD_geom_config.json',  # Detector configuration file
    'data_file': base_dir_path() + 'data/water/muon/50_data_like_events.root',  # ROOT file with reference photons
    'detector_type': 'Cylinder',  # Detector geometry type
    'entry_idx': 0,  # Which entry to use from ROOT file
    'n_photons': 500_000,  # Number of photons to simulate
    'K': 6,  # Number of scattering iterations
    'seed': 12345,  # Random seed
    'min_charge': 5.0,  # Minimum charge threshold for display
    'color_by': 'charge',  # Color sensor hits by 'charge' or 'time'
    'dark_theme': True,  # Use dark theme for disc visualizations
    'log_scale': False,  # Use log scale for disc visualizations
    'save_figures': True  # Save figures to files
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Setup Detector and Simulators

In [ ]:
# Setup detector
print("Setting up detector...")
detector = generate_detector(CONFIG['detector_config'])
sensor_positions = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)
n_sensors = len(sensor_positions)

print(f"  Type: {CONFIG['detector_type']}")
print(f"  Sensors: {n_sensors:,}")
print(f"  Bounds: {detector_bounds}")

# Sensor parameters
sensor_params = (
    jnp.array(50.0),    # scatter_length
    jnp.array(0.1),     # reflection_rate
    jnp.array(100.0),   # absorption_length
    jnp.array(0.001)    # gumbel_softmax_temperature
)

In [ ]:
# Setup simulators
print("Setting up simulators...")

# Prediction simulator (regular physics simulation)
prediction_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.05,
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=False
)

# Data simulator (transforms reference photons)
data_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,  # Zero temperature for data mode
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=True
)

print("  Simulators ready")

## Load Reference Photons and Generate Track Parameters

In [ ]:
# Load photon data from ROOT file
print(f"Loading reference photons from ROOT file...")
photon_data = read_photon_data_from_photonsim(CONFIG['data_file'], CONFIG['entry_idx'])
photon_data['N'] = len(photon_data['photon_origins'])

print(f"  Number of photons: {photon_data['N']:,}")
print(f"  Primary energy: {photon_data['energy']:.1f} MeV")

# Generate track parameters
print("\nGenerating track parameters...")
key = jax.random.PRNGKey(CONFIG['seed'])
track_position, track_direction, _ = generate_random_event_params(key, detector_bounds)
track_energy = photon_data['energy']

print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
print(f"  Energy: {track_energy:.1f} MeV")

## Simulate Events

In [ ]:
# Simulate events
print("Simulating events...")
event_key = jax.random.PRNGKey(CONFIG['seed'] + 1000)

# Prediction-like event
print("  Generating prediction-like event...")
# Convert direction for prediction simulator
theta = jnp.arccos(jnp.clip(track_direction[2], -1.0, 1.0))
phi = jnp.arctan2(track_direction[1], track_direction[0])
direction_angles = jnp.array([theta, phi])

prediction_params = (track_energy, track_position, direction_angles)
prediction_charges, prediction_times = prediction_simulator(prediction_params, sensor_params, event_key)

# Data-like event
print("  Generating data-like event...")
data_params = (track_energy, track_position, track_direction)
data_charges, data_times = data_simulator(data_params, sensor_params, event_key, photon_data)

print("  Events generated successfully")

## Event Analysis and Statistics

In [ ]:
def analyze_events(prediction_charges, prediction_times, data_charges, data_times, 
                  track_position, track_direction, track_energy, min_charge=5.0):
    """
    Perform quantitative analysis of the two event types.
    """
    # Filter active sensors
    pred_active = prediction_charges > min_charge
    data_active = data_charges > min_charge
    
    pred_charges_active = prediction_charges[pred_active]
    pred_times_active = prediction_times[pred_active]
    data_charges_active = data_charges[data_active]
    data_times_active = data_times[data_active]
    
    print("Event Analysis Summary")
    print("=" * 50)
    print(f"Track Parameters:")
    print(f"  Energy: {track_energy:.1f} MeV")
    print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
    print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
    print()
    
    print(f"Prediction-like Event:")
    print(f"  Active sensors: {len(pred_charges_active):,}")
    print(f"  Total charge: {np.sum(pred_charges_active):.1f}")
    print(f"  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}")
    print(f"  Charge range: [{np.min(pred_charges_active):.1f}, {np.max(pred_charges_active):.1f}]")
    print(f"  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns")
    print(f"  Time range: [{np.min(pred_times_active):.1f}, {np.max(pred_times_active):.1f}] ns")
    print()
    
    print(f"Data-like Event:")
    print(f"  Active sensors: {len(data_charges_active):,}")
    print(f"  Total charge: {np.sum(data_charges_active):.1f}")
    print(f"  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}")
    print(f"  Charge range: [{np.min(data_charges_active):.1f}, {np.max(data_charges_active):.1f}]")
    print(f"  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns")
    print(f"  Time range: [{np.min(data_times_active):.1f}, {np.max(data_times_active):.1f}] ns")
    print()
    
    # Comparison
    sensor_ratio = len(data_charges_active) / len(pred_charges_active) if len(pred_charges_active) > 0 else 0
    charge_ratio = np.sum(data_charges_active) / np.sum(pred_charges_active) if np.sum(pred_charges_active) > 0 else 0
    
    print(f"Comparison:")
    print(f"  Active sensor ratio (data/prediction): {sensor_ratio:.2f}")
    print(f"  Total charge ratio (data/prediction): {charge_ratio:.2f}")
    
    # Common sensors
    pred_indices = np.where(pred_active)[0]
    data_indices = np.where(data_active)[0]
    common_active = set(pred_indices) & set(data_indices)
    n_common = len(common_active)
    print(f"  Sensors active in both: {n_common:,}")
    print(f"  Overlap fraction: {n_common / max(len(pred_charges_active), len(data_charges_active)):.2f}")
    
    return pred_indices, data_indices

# Perform analysis
prediction_indices, data_indices = analyze_events(
    prediction_charges, prediction_times, data_charges, data_times,
    track_position, track_direction, track_energy, CONFIG['min_charge']
)

## 3D Scatter Plot Visualizations

Create interactive 3D scatter plots with track visualization:

In [ ]:
def create_3d_event_plot(charges, times, sensor_positions, track_position, track_direction, 
                        track_energy, detector_bounds, title, color_by='charge', min_charge=5.0):
    """
    Create a 3D visualization of an event using scatter plots.
    """
    # Filter hits with significant charge
    significant_mask = charges > min_charge
    hit_positions = sensor_positions[significant_mask]
    hit_charges = charges[significant_mask]
    hit_times = times[significant_mask]
    
    if len(hit_positions) == 0:
        print(f"Warning: No hits above threshold {min_charge} for {title}")
        return None
    
    # Setup color data
    if color_by == 'charge':
        color_data = hit_charges
        color_label = 'Charge'
        colorscale = 'viridis'
    else:
        color_data = hit_times
        color_label = 'Time [ns]'
        colorscale = 'plasma'
    
    # Create figure
    fig = go.Figure()
    
    # Add sensor hits
    fig.add_trace(go.Scatter3d(
        x=hit_positions[:, 0],
        y=hit_positions[:, 1], 
        z=hit_positions[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=color_data,
            colorscale=colorscale,
            opacity=0.8,
            colorbar=dict(title=color_label)
        ),
        name=f'Sensor Hits ({len(hit_positions)})',
        text=[f'Charge: {c:.1f}<br>Time: {t:.1f} ns' for c, t in zip(hit_charges, hit_times)],
        hovertemplate='%{text}<extra></extra>'
    ))
    
    # Add track origin
    fig.add_trace(go.Scatter3d(
        x=[track_position[0]],
        y=[track_position[1]],
        z=[track_position[2]],
        mode='markers',
        marker=dict(size=10, color='red', symbol='diamond'),
        name='Track Origin',
        text=f'Position: [{track_position[0]:.2f}, {track_position[1]:.2f}, {track_position[2]:.2f}]',
        hovertemplate='%{text}<extra></extra>'
    ))
    
    # Add track direction line
    track_length = 8.0  # meters
    track_end = track_position + track_length * track_direction
    
    fig.add_trace(go.Scatter3d(
        x=[track_position[0], track_end[0]],
        y=[track_position[1], track_end[1]],
        z=[track_position[2], track_end[2]],
        mode='lines',
        line=dict(color='red', width=6),
        name='Track Direction',
        text=f'Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]',
        hovertemplate='%{text}<extra></extra>'
    ))
    
    # Add detector boundary (simplified)
    if detector_bounds['type'] == 'cylinder':
        # Create cylinder outline
        theta_vals = np.linspace(0, 2*np.pi, 50)
        r = detector_bounds['r']
        h = detector_bounds['H']
        
        # Top and bottom circles
        for z_val in [-h/2, h/2]:
            fig.add_trace(go.Scatter3d(
                x=r * np.cos(theta_vals),
                y=r * np.sin(theta_vals),
                z=np.full_like(theta_vals, z_val),
                mode='lines',
                line=dict(color='gray', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Set layout
    max_extent = max(detector_bounds.get('r', 10), detector_bounds.get('H', 20)/2) * 1.2
    
    fig.update_layout(
        title=dict(
            text=f'{title}<br>Energy: {track_energy:.1f} MeV, Active Sensors: {len(hit_positions)}',
            x=0.5
        ),
        scene=dict(
            xaxis=dict(title='X [m]', range=[-max_extent, max_extent]),
            yaxis=dict(title='Y [m]', range=[-max_extent, max_extent]),
            zaxis=dict(title='Z [m]', range=[-max_extent, max_extent]),
            aspectmode='cube'
        ),
        width=800,
        height=700,
        showlegend=True
    )
    
    return fig

In [ ]:
# Create individual scatter plots
print("Creating scatter plot visualizations...")

# Prediction-like event
fig_pred = create_3d_event_plot(
    prediction_charges, prediction_times, sensor_positions,
    track_position, track_direction, track_energy, detector_bounds,
    title="Prediction-like Event (Physics Simulation)",
    color_by=CONFIG['color_by'], min_charge=CONFIG['min_charge']
)

if fig_pred:
    fig_pred.show()
    if CONFIG['save_figures']:
        detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        # Use base_dir_path for figures directory
        figures_dir = Path(base_dir_path()) / 'figures'
        figures_dir.mkdir(exist_ok=True)
        filename = figures_dir / f'{detector_name}_prediction_scatter_{timestamp}.html'
        fig_pred.write_html(str(filename))
        print(f"Saved: {filename}")

In [ ]:
# Data-like event
fig_data = create_3d_event_plot(
    data_charges, data_times, sensor_positions,
    track_position, track_direction, track_energy, detector_bounds,
    title="Data-like Event (Reference Photon Transformation)",
    color_by=CONFIG['color_by'], min_charge=CONFIG['min_charge']
)

if fig_data:
    fig_data.show()
    if CONFIG['save_figures']:
        figures_dir = Path(base_dir_path()) / 'figures'
        filename = figures_dir / f'{detector_name}_data_scatter_{timestamp}.html'
        fig_data.write_html(str(filename))
        print(f"Saved: {filename}")

In [ ]:
# Side-by-side comparison
def create_comparison_plot(prediction_charges, prediction_times, data_charges, data_times,
                          sensor_positions, track_position, track_direction, track_energy,
                          detector_bounds, color_by='charge', min_charge=5.0):
    """
    Create side-by-side comparison of prediction-like and data-like events.
    """
    # Create subplots
    fig = sp.make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=('Prediction-like Event', 'Data-like Event'),
        horizontal_spacing=0.05
    )
    
    # Helper function to add event to subplot
    def add_event_to_subplot(charges, times, col, event_type):
        # Filter hits
        significant_mask = charges > min_charge
        hit_positions = sensor_positions[significant_mask]
        hit_charges = charges[significant_mask]
        hit_times = times[significant_mask]
        
        if len(hit_positions) == 0:
            return
        
        # Color data
        color_data = hit_charges if color_by == 'charge' else hit_times
        colorscale = 'viridis' if color_by == 'charge' else 'plasma'
        
        # Add sensor hits
        fig.add_trace(go.Scatter3d(
            x=hit_positions[:, 0],
            y=hit_positions[:, 1],
            z=hit_positions[:, 2],
            mode='markers',
            marker=dict(
                size=3,
                color=color_data,
                colorscale=colorscale,
                opacity=0.7
            ),
            name=f'{event_type} Hits',
            showlegend=True,
            text=[f'Charge: {c:.1f}<br>Time: {t:.1f} ns' for c, t in zip(hit_charges, hit_times)],
            hovertemplate='%{text}<extra></extra>'
        ), row=1, col=col)
        
        # Add track origin
        fig.add_trace(go.Scatter3d(
            x=[track_position[0]],
            y=[track_position[1]],
            z=[track_position[2]],
            mode='markers',
            marker=dict(size=8, color='red', symbol='diamond'),
            name=f'{event_type} Origin',
            showlegend=False
        ), row=1, col=col)
        
        # Add track direction
        track_end = track_position + 6.0 * track_direction
        fig.add_trace(go.Scatter3d(
            x=[track_position[0], track_end[0]],
            y=[track_position[1], track_end[1]],
            z=[track_position[2], track_end[2]],
            mode='lines',
            line=dict(color='red', width=5),
            name=f'{event_type} Track',
            showlegend=False
        ), row=1, col=col)
    
    # Add both events
    add_event_to_subplot(prediction_charges, prediction_times, 1, 'Prediction')
    add_event_to_subplot(data_charges, data_times, 2, 'Data')
    
    # Update layout
    max_extent = max(detector_bounds.get('r', 10), detector_bounds.get('H', 20)/2) * 1.2
    
    scene_dict = dict(
        xaxis=dict(title='X [m]', range=[-max_extent, max_extent]),
        yaxis=dict(title='Y [m]', range=[-max_extent, max_extent]),
        zaxis=dict(title='Z [m]', range=[-max_extent, max_extent]),
        aspectmode='cube'
    )
    
    fig.update_layout(
        title=dict(
            text=f'Event Comparison - Energy: {track_energy:.1f} MeV',
            x=0.5
        ),
        scene=scene_dict,
        scene2=scene_dict,
        width=1600,
        height=700,
        showlegend=True
    )
    
    return fig

# Create comparison plot
fig_comp = create_comparison_plot(
    prediction_charges, prediction_times, data_charges, data_times,
    sensor_positions, track_position, track_direction, track_energy,
    detector_bounds, color_by=CONFIG['color_by'], min_charge=CONFIG['min_charge']
)

fig_comp.show()
if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    filename = figures_dir / f'{detector_name}_comparison_scatter_{timestamp}.html'
    fig_comp.write_html(str(filename))
    print(f"Saved: {filename}")

## 3D Disc-based Visualizations

Use the detector's native visualization method for disc-based plots:

In [ ]:
# Visualization parameters for disc plots
surface_color = 'black' if CONFIG['dark_theme'] else 'gray'
colorscale_charge = 'inferno' if CONFIG['dark_theme'] else 'viridis'
colorscale_time = 'plasma'

# Convert our dense arrays to sparse format (like load_single_event does)
# The working notebook uses sparse format: indices, charges[indices], times[indices]

# Find non-zero indices for prediction event
pred_nonzero_mask = prediction_charges > 0
pred_sparse_indices = np.where(pred_nonzero_mask)[0]
pred_sparse_charges = prediction_charges[pred_sparse_indices]
pred_sparse_times = prediction_times[pred_sparse_indices]

# Find non-zero indices for data event  
data_nonzero_mask = data_charges > 0
data_sparse_indices = np.where(data_nonzero_mask)[0]
data_sparse_charges = data_charges[data_sparse_indices]
data_sparse_times = data_times[data_sparse_indices]

print("Creating disc-based visualizations...")
print(f"Prediction event: {len(pred_sparse_indices)} sensors with non-zero charge")
print(f"Data event: {len(data_sparse_indices)} sensors with non-zero charge")
print(f"Prediction charges range: [{np.min(pred_sparse_charges):.2f}, {np.max(pred_sparse_charges):.2f}]")
print(f"Data charges range: [{np.min(data_sparse_charges):.2f}, {np.max(data_sparse_charges):.2f}]")
print("\n=== Prediction-like Event - Charge ===\n")

In [ ]:
# Prediction event - Charge visualization
# Now use the sparse format exactly like the working notebook
detector.visualize_event_data_plotly_discs(
    pred_sparse_indices, 
    pred_sparse_charges, 
    pred_sparse_times,
    show_all_sensors=True,
    log_scale=CONFIG['log_scale'],
    show_colorbar=True,
    dark_theme=CONFIG['dark_theme'],
    plot_time=False,
    colorscale=colorscale_charge,
    surface_color=surface_color,
    title="Prediction-like Event - Charge"
)

In [ ]:
print("\n=== Data-like Event - Charge ===\n")

# Data event - Charge visualization
detector.visualize_event_data_plotly_discs(
    data_sparse_indices, 
    data_sparse_charges, 
    data_sparse_times,
    show_all_sensors=True,
    log_scale=CONFIG['log_scale'],
    show_colorbar=True,
    dark_theme=CONFIG['dark_theme'],
    plot_time=False,
    colorscale=colorscale_charge,
    surface_color=surface_color,
    title="Data-like Event - Charge"
)

In [ ]:
print("\n=== Prediction-like Event - Time ===\n")

# Prediction event - Time visualization
detector.visualize_event_data_plotly_discs(
    pred_sparse_indices, 
    pred_sparse_charges, 
    pred_sparse_times,
    show_all_sensors=True,
    log_scale=False,  # Time shouldn't use log scale
    show_colorbar=True,
    dark_theme=CONFIG['dark_theme'],
    plot_time=True,
    colorscale=colorscale_time,
    surface_color=surface_color,
    title="Prediction-like Event - Time"
)

In [ ]:
print("\n=== Data-like Event - Time ===\n")

# Data event - Time visualization
detector.visualize_event_data_plotly_discs(
    data_sparse_indices, 
    data_sparse_charges, 
    data_sparse_times,
    show_all_sensors=True,
    log_scale=False,  # Time shouldn't use log scale
    show_colorbar=True,
    dark_theme=CONFIG['dark_theme'],
    plot_time=True,
    colorscale=colorscale_time,
    surface_color=surface_color,
    title="Data-like Event - Time"
)

## Statistical Comparison Plots

In [ ]:
# Create statistical comparison plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Filter active sensors
pred_active = prediction_charges > CONFIG['min_charge']
data_active = data_charges > CONFIG['min_charge']

pred_charges_active = prediction_charges[pred_active]
pred_times_active = prediction_times[pred_active]
data_charges_active = data_charges[data_active]
data_times_active = data_times[data_active]

# Charge distributions
ax1.hist(pred_charges_active, bins=50, alpha=0.7, label='Prediction-like', color='blue', density=True)
ax1.hist(data_charges_active, bins=50, alpha=0.7, label='Data-like', color='red', density=True)
ax1.set_xlabel('Charge')
ax1.set_ylabel('Density')
ax1.set_title('Charge Distribution Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time distributions
ax2.hist(pred_times_active, bins=50, alpha=0.7, label='Prediction-like', color='blue', density=True)
ax2.hist(data_times_active, bins=50, alpha=0.7, label='Data-like', color='red', density=True)
ax2.set_xlabel('Time [ns]')
ax2.set_ylabel('Density')
ax2.set_title('Time Distribution Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Charge vs Time scatter
ax3.scatter(pred_charges_active, pred_times_active, alpha=0.6, s=10, label='Prediction-like', color='blue')
ax3.scatter(data_charges_active, data_times_active, alpha=0.6, s=10, label='Data-like', color='red')
ax3.set_xlabel('Charge')
ax3.set_ylabel('Time [ns]')
ax3.set_title('Charge vs Time Correlation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Summary statistics
stats_text = f"""
Prediction-like Event:
  Active sensors: {len(pred_charges_active):,}
  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}
  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns

Data-like Event:
  Active sensors: {len(data_charges_active):,}
  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}
  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns

Track Parameters:
  Energy: {track_energy:.1f} MeV
  Position: [{track_position[0]:.2f}, {track_position[1]:.2f}, {track_position[2]:.2f}] m
  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]
"""

ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=10, 
         verticalalignment='top', fontfamily='monospace')
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')
ax4.set_title('Event Statistics')

plt.tight_layout()
plt.show()

if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    filename = figures_dir / f'{detector_name}_statistical_comparison_{timestamp}.png'
    plt.savefig(str(filename), dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")

## Summary

This notebook has successfully generated comprehensive 3D visualizations comparing data-like and prediction-like events:

### Generated Visualizations:
1. **3D Scatter Plots**: Interactive plots with track visualization
   - Individual plots for each event type
   - Side-by-side comparison
   - Configurable coloring by charge or time

2. **3D Disc Plots**: Using detector's native visualization method
   - Charge-based visualizations for both event types
   - Time-based visualizations for both event types
   - Proper handling of inactive sensors

3. **Statistical Analysis**: 
   - Distribution comparisons
   - Correlation analysis
   - Quantitative event metrics

### Key Insights:
- **Event characteristics**: Data-like events typically show different charge and time distributions compared to prediction-like events
- **Sensor activity patterns**: The overlap and differences in activated sensors provide insights into detector response
- **Track reconstruction**: Both event types use the same track parameters, enabling direct comparison of detector response

### Files Saved:
- HTML files for interactive 3D scatter plots
- PNG file for statistical comparison plots
- All saved in the `figures/` directory with timestamp

To customize the analysis, modify the `CONFIG` dictionary at the beginning of the notebook.